# Perguntas com retrieval TF-IDF

Notebook mínimo para consultar por TF-IDF os chunks já armazenados por `rag.ipynb`. Execute o notebook principal ao menos uma vez antes deste.

In [1]:
import os
from pathlib import Path
from pprint import pprint

import chromadb
from dotenv import load_dotenv
from google import genai

import rag_utils

PROJECT_ROOT = Path.cwd()
load_dotenv(PROJECT_ROOT / ".env")
API_KEY = os.environ.get("GEMINI_API_KEY")
assert API_KEY, "Defina GEMINI_API_KEY no arquivo .env da raiz do projeto."

# Modelo e banco
client = genai.Client(api_key=API_KEY)
database = chromadb.PersistentClient(path=str(PROJECT_ROOT / "db" / "chroma"))
collection = database.get_collection(name="regulamento_ufrpe_v1")
assert collection.count() > 0, "Índice vazio: execute rag.ipynb primeiro."

records = rag_utils.records_from_collection(collection)
retriever = rag_utils.TfidfRetriever(records)
rag = rag_utils.RAGAssistant(client, retriever, generation_model="gemini-flash-latest")

In [2]:
QUESTION = "Quais documentos o regulamento prevê que a UFRPE pode expedir para o(a) discente?"
result = rag.ask(QUESTION, k=5)

pprint(rag_utils.hits_as_rows(result.hits), sort_dicts=False)
print(f"\nResposta:\n{result.answer}")

[{'rank': 1,
  'tfidf_cosine_similarity': 0.2321,
  'paper': '25_documentos_e_nome_social',
  'page': 1,
  'chunk': 0,
  'snippet': 'UFRPE — Regulamento Geral da Graduação (PREG, 2024)\n'
             'Documentos e registros oficiais e nome social\n'
             'TÍTULO XIII\n'
             'DOS DOCUMENTOS E REGISTROS OFICIAIS\n'
             'Art. 340. Os documentos oficiais relativos à graduação são de '
             'dois tipos:\n'
             '…'},
 {'rank': 2,
  'tfidf_cosine_similarity': 0.1733,
  'paper': '29_disposicoes_finais_e_anexos',
  'page': 2,
  'chunk': 0,
  'snippet': 'todos os componentes curriculares nos quais o discente acumulou '
             'carga horária após o início do curso,\n'
             'incluindo-se os componentes curriculares incorporados após o '
             'início do curso e excluindo-se os\n'
             'componentes curriculare…'},
 {'rank': 3,
  'tfidf_cosine_similarity': 0.1713,
  'paper': '28_vivencias_e_atividades_interdisciplinares',
  'pa